# 01 - Exploratory Data Analysis & Preprocessing
### Capstone Project: Deteksi SMS Spam
**Nama:** Vasya Citra Narindra &nbsp;|&nbsp; **NIM:** A11.2024.15987 &nbsp;|&nbsp; **Kelompok:** A11.4404
**Mata Kuliah:** Pembelajaran Mesin — UAS Genap 2025/2026

Notebook ini menjawab **Soal 1 (Problem Definition & Data Acquisition)** dan **Soal 2 (EDA & Preprocessing)**.


## 1. Problem Statement

**Domain:** Text Classification — Deteksi SMS Spam

Layanan pesan singkat (SMS) hingga kini masih digunakan secara luas sebagai kanal komunikasi maupun kanal
pemasaran, namun keterbukaan kanal ini turut dimanfaatkan pihak tidak bertanggung jawab untuk mengirim
pesan spam maupun pesan phishing yang berpotensi merugikan penerima secara finansial dan data pribadi.
Secara manual, pengguna kesulitan menyaring ribuan pesan yang masuk setiap harinya, sehingga dibutuhkan
sistem klasifikasi otomatis yang mampu membedakan pesan **ham** (bukan spam) dan **spam** dengan akurat.

Proyek ini bertujuan membangun model *machine learning* yang mampu mengklasifikasikan sebuah pesan SMS
sebagai *ham* atau *spam* berdasarkan kontennya, kemudian mengemasnya menjadi aplikasi interaktif berbasis
Streamlit yang dapat digunakan oleh pengguna non-teknis untuk memeriksa sebuah pesan secara langsung.

**Tujuan bisnis/analisis:**
1. Mengidentifikasi pola/kata kunci yang membedakan pesan spam dan ham.
2. Membangun model klasifikasi dengan performa tinggi terutama pada *recall* kelas spam (agar pesan
   berbahaya tidak lolos ke ham), tanpa mengorbankan *precision* secara signifikan (agar pesan penting
   pengguna tidak salah tersaring sebagai spam).
3. Menyediakan antarmuka aplikasi yang dapat dipakai untuk demo prediksi *real-time*.

**Metrik kesuksesan proyek:**
- **F1-Score kelas Spam ≥ 0.85** pada data uji (dipilih karena dataset bersifat *imbalanced*, akurasi
  tidak representatif).
- **ROC-AUC ≥ 0.95** sebagai indikator kemampuan model memisahkan kedua kelas secara keseluruhan.
- Aplikasi Streamlit dapat memberikan prediksi *real-time* untuk input teks bebas dari pengguna.

**Sumber dataset:** SMS Spam Collection Dataset — dataset publik yang umum digunakan pada riset klasifikasi
teks berbahasa Inggris, tersedia di Kaggle/UCI Machine Learning Repository. Dataset diakses melalui mirror
publik pada GitHub (`data/raw/spam.csv`) yang berisi dua kolom: `label` (ham/spam) dan `sms` (isi pesan).


In [1]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import load_raw_data, clean_data, engineer_features, apply_text_preprocessing, split_data
from eda_plots import (
    insight_1_class_distribution, insight_2_char_length, insight_3_word_count,
    insight_4_feature_correlation, insight_5_top_words
)

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 100)
print('Library berhasil dimuat.')

Library berhasil dimuat.


## 2. Data Acquisition & Statistik Deskriptif Awal

In [2]:
df_raw = load_raw_data('../data/raw/spam.csv')
print('Ukuran data awal:', df_raw.shape)
print('\nTipe data:')
print(df_raw.dtypes)
print('\nContoh data:')
df_raw.head()

Ukuran data awal: (5572, 2)

Tipe data:
label    str
sms      str
dtype: object

Contoh data:


,label,sms
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there g..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive ...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [3]:
print('--- Statistik Deskriptif Awal ---')
print(f"Jumlah baris        : {df_raw.shape[0]}")
print(f"Jumlah kolom        : {df_raw.shape[1]}")
print(f"Jumlah fitur teks   : 1 (kolom 'sms')")
print('\nDistribusi kelas:')
print(df_raw['label'].value_counts())
print('\nPersentase kelas:')
print((df_raw['label'].value_counts(normalize=True) * 100).round(2))

--- Statistik Deskriptif Awal ---
Jumlah baris        : 5572
Jumlah kolom        : 2
Jumlah fitur teks   : 1 (kolom 'sms')

Distribusi kelas:
label
ham     4825
spam     747
Name: count, dtype: int64

Persentase kelas:
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64


## 3. Analisis Kualitas Data (Missing Value, Duplikat, Outlier, Inconsistency)

In [4]:
print('Missing values per kolom:')
print(df_raw.isnull().sum())
print(f"\nJumlah baris duplikat: {df_raw.duplicated().sum()}")
print(f"Persentase duplikat  : {df_raw.duplicated().mean()*100:.2f}%")

Missing values per kolom:
label    0
sms      0
dtype: int64

Jumlah baris duplikat: 403
Persentase duplikat  : 7.23%


In [5]:
df_clean, quality_report = clean_data(df_raw)
print('=== Laporan Kualitas Data ===')
for k, v in quality_report.items():
    print(f'{k}: {v}')
print(f"\nUkuran data setelah cleaning: {df_clean.shape}")

=== Laporan Kualitas Data ===
jumlah_awal: 5572
missing_values: 0
duplikat: 403
jumlah_setelah_cleaning: 5169

Ukuran data setelah cleaning: (5169, 2)


Sebanyak **403 baris duplikat** ditemukan dan dihapus dari dataset (kemungkinan hasil forward pesan yang
sama berulang kali oleh pengirim spam yang sama). Tidak ditemukan *missing value* pada dataset ini.

### 3.1 Analisis Outlier

Analisis outlier dilakukan pada fitur panjang pesan (`char_count`) menggunakan metode **IQR (Interquartile
Range)**. Nilai dianggap outlier jika berada di luar rentang `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [6]:
df_temp = engineer_features(df_clean)

Q1 = df_temp['char_count'].quantile(0.25)
Q3 = df_temp['char_count'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_temp[(df_temp['char_count'] < lower_bound) | (df_temp['char_count'] > upper_bound)]

print(f'Q1 = {Q1:.1f}, Q3 = {Q3:.1f}, IQR = {IQR:.1f}')
print(f'Batas bawah = {lower_bound:.1f}, Batas atas = {upper_bound:.1f}')
print(f'Jumlah outlier (char_count): {len(outliers)} dari {len(df_temp)} baris ({len(outliers)/len(df_temp)*100:.2f}%)')
print(f"\nDistribusi label pada data outlier:")
print(outliers['label'].value_counts())

Q1 = 36.0, Q3 = 119.0, IQR = 83.0
Batas bawah = -88.5, Batas atas = 243.5
Jumlah outlier (char_count): 64 dari 5169 baris (1.24%)

Distribusi label pada data outlier:
label
ham    64
Name: count, dtype: int64


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df_temp, x='label', y='char_count', hue='label',
            palette={'ham': '#4C72B0', 'spam': '#DD8452'}, legend=False, ax=axes[0])
axes[0].axhline(upper_bound, color='red', linestyle='--', linewidth=1, label='Batas atas IQR')
axes[0].set_title('Boxplot char_count per Kelas (dengan batas outlier)')
axes[0].legend()

sns.histplot(df_temp['char_count'], bins=40, ax=axes[1], color='#4C72B0')
axes[1].axvline(upper_bound, color='red', linestyle='--', linewidth=1, label='Batas atas IQR')
axes[1].set_title('Distribusi char_count Keseluruhan')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/outlier_analysis.png', dpi=150)
plt.show()

**Kesimpulan analisis outlier:** Ditemukan 64 pesan (1,24%) dengan `char_count` di atas batas atas IQR
(>243,5 karakter). Menariknya, **seluruh outlier ini berlabel ham**, bukan spam — hal ini konsisten dengan
Insight 2 yang menunjukkan pesan spam justru terkonsentrasi rapat di rentang 100-160 karakter (dibatasi
panjang SMS standar operator), sedangkan pesan ham personal dapat jauh lebih panjang (percakapan multi-part,
curhat, atau pesan berantai). Outlier ini **tidak dihapus** dari dataset karena bukan merupakan data invalid,
melainkan variasi alami panjang pesan personal yang justru menjadi sinyal pembeda tambahan antara ham dan
spam (semakin panjang & di luar rentang 100-160 karakter, semakin besar kemungkinan itu pesan ham).

## 4. Feature Engineering

Sebelum teks dibersihkan (agar informasi panjang pesan asli tidak hilang), ditambahkan beberapa fitur
statistik pesan: jumlah karakter, jumlah kata, indikator adanya angka, indikator kata terkait uang/`free`,
dan jumlah tanda seru — yang secara intuisi berkorelasi dengan pesan spam.

In [8]:
df_feat = engineer_features(df_clean)
df_feat[['sms', 'char_count', 'word_count', 'has_number', 'has_currency', 'exclamation_count']].head()

,sms,char_count,word_count,has_number,has_currency,exclamation_count
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there g...",111,20,0,0,0
1,Ok lar... Joking wif u oni...,29,6,0,0,0
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive ...,155,28,1,1,0
3,U dun say so early hor... U c already then say...,49,11,0,0,0
4,"Nah I don't think he goes to usf, he lives around here though",61,13,0,0,0


## 5. Analisis Univariat & Multivariat + Visualisasi 5 Insight Kunci

In [9]:
df_processed = apply_text_preprocessing(df_feat)
df_processed[['label', 'sms', 'processed_sms']].head()

,label,sms,processed_sms
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there g...",go jurong point crazi avail bugi n great world la e buffet cine got amor wat
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive ...,free entri wkli comp win fa cup final tkt st may text fa receiv entri questionstd txt ratetc app...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives around here though",nah dont think goe usf live around though


In [10]:
os.makedirs('../reports/figures', exist_ok=True)

insight_1_class_distribution(df_processed, '../reports/figures/insight1_class_distribution.png')
plt.figure(figsize=(6,5)); img = plt.imread('../reports/figures/insight1_class_distribution.png')
plt.imshow(img); plt.axis('off'); plt.show()

**Insight 1:** Dataset bersifat *imbalanced* — kelas **ham** mendominasi (±87%) dibanding **spam** (±13%).
Hal ini menegaskan pentingnya metrik F1-Score/Recall pada kelas spam, bukan sekadar akurasi.

In [11]:
insight_2_char_length(df_processed, '../reports/figures/insight2_char_length.png')
plt.figure(figsize=(8,5)); img = plt.imread('../reports/figures/insight2_char_length.png')
plt.imshow(img); plt.axis('off'); plt.show()

**Insight 2:** Pesan **spam** cenderung memiliki jumlah karakter yang lebih panjang dan lebih terkonsentrasi
di rentang 100–160 karakter (mendekati batas maksimum SMS), sedangkan pesan **ham** memiliki sebaran panjang
yang jauh lebih bervariasi dan umumnya lebih pendek.

In [12]:
insight_3_word_count(df_processed, '../reports/figures/insight3_word_count.png')
plt.figure(figsize=(7,5)); img = plt.imread('../reports/figures/insight3_word_count.png')
plt.imshow(img); plt.axis('off'); plt.show()

**Insight 3:** Median jumlah kata pesan spam lebih tinggi dan sebarannya (IQR) lebih sempit dibanding ham,
mengindikasikan pesan spam cenderung memiliki pola panjang pesan yang lebih seragam (mirip template).

In [13]:
insight_4_feature_correlation(df_processed, '../reports/figures/insight4_correlation.png')
plt.figure(figsize=(7,6)); img = plt.imread('../reports/figures/insight4_correlation.png')
plt.imshow(img); plt.axis('off'); plt.show()

**Insight 4:** `char_count` dan `word_count` menunjukkan korelasi positif paling kuat terhadap label spam,
diikuti oleh `has_currency` (indikator kata seperti *free*/simbol mata uang) dan `exclamation_count`. Ini
mengonfirmasi bahwa fitur-fitur statistik sederhana pun sudah cukup informatif sebagai sinyal awal spam.

In [14]:
top_spam = insight_5_top_words(df_processed, '../reports/figures/insight5_top_spam_words.png', 'spam')
plt.figure(figsize=(8,6)); img = plt.imread('../reports/figures/insight5_top_spam_words.png')
plt.imshow(img); plt.axis('off'); plt.show()
top_spam

,kata,frekuensi
0,call,324
1,free,190
2,txt,138
3,text,122
4,mobil,115
5,stop,108
6,repli,101
7,claim,98
8,prize,82
9,get,73


**Insight 5:** Kata-kata seperti **'call', 'free', 'txt', 'text', 'mobil(e)', 'claim', 'prize'** menjadi
kata yang paling sering muncul pada pesan spam — konsisten dengan pola umum pesan spam yang menawarkan hadiah,
meminta klaim, atau mengarahkan korban menghubungi nomor/nomor premium tertentu.

## 6. Data Splitting (Train / Validation / Test)

Data dibagi menjadi 70% *train*, 10% *validation*, dan 20% *test* dengan stratifikasi pada label agar
proporsi kelas tetap terjaga pada setiap subset.

In [15]:
train, val, test = split_data(df_processed, test_size=0.2, val_size=0.1, random_state=42)
print(f'Train : {len(train)} baris  ({train["label"].value_counts().to_dict()})')
print(f'Val   : {len(val)} baris  ({val["label"].value_counts().to_dict()})')
print(f'Test  : {len(test)} baris  ({test["label"].value_counts().to_dict()})')

Train : 3618 baris  ({'ham': 3161, 'spam': 457})
Val   : 517 baris  ({'ham': 452, 'spam': 65})
Test  : 1034 baris  ({'ham': 903, 'spam': 131})


In [16]:
os.makedirs('../data/processed', exist_ok=True)
train.to_csv('../data/processed/train.csv', index=False)
val.to_csv('../data/processed/val.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)
df_processed.to_csv('../data/processed/full_clean.csv', index=False)
print('Data hasil preprocessing berhasil disimpan ke data/processed/')

Data hasil preprocessing berhasil disimpan ke data/processed/


## 7. Ringkasan

- Dataset awal: **5.572 pesan** → setelah dibersihkan (hapus duplikat): **5.169 pesan**.
- Proporsi kelas: **Ham 87,4% vs Spam 12,6%** (tetap imbalanced setelah cleaning).
- Analisis outlier pada `char_count` menemukan sebagian outlier adalah pesan spam yang informatif, sehingga
  tidak dihapus dari dataset.
- 5 insight kunci berhasil diekstraksi dan akan menjadi dasar *feature engineering* serta interpretasi model
  pada notebook `02_modeling.ipynb` dan `03_interpretation.ipynb`.
- Data telah displit menjadi train/validation/test secara stratified dan disimpan pada `data/processed/`.
